# Week 4, Day 5 — The Sidekick
### Local Models Edition — by Abhishek

The project the whole week has built toward. The Sidekick is a personal
co-worker you give a task and a definition of success — it works away using
a real browser, a sandboxed filesystem, web search and Wikipedia, until it
meets your criteria or needs to ask you something.

**Fully local and free:** the worker model is Ollama or Hugging Face, search
is DuckDuckGo, notifications default to a local log file, and the browser +
filesystem tools are free MCP subprocesses launched via `npx` — no API key
anywhere in this lab.


## How it is built

The worker is one `create_agent` (Layer 3) with tools and a stack of
middleware:

- `TolerateToolErrors` (a small custom middleware) hands tool failures back
  to the model as a message instead of crashing the run — real-world tools
  like a browser fail sometimes
- `TodoListMiddleware` gives it a plan it keeps updated, surfaced in the UI
- `PIIMiddleware` redacts emails from the model's messages, and credit card
  numbers from tool results
- `ModelCallLimitMiddleware` caps a run at 30 model calls, so a lost agent
  can't loop forever — this matters *more* on local models, which are more
  prone to getting stuck in unproductive tool-call loops than frontier models
- `HumanInTheLoopMiddleware` pauses for your approval before a notification
  goes out, or before the agent asks you for help it can't give itself
  (`request_human_help` — for logins, captchas, 2FA)

Around the worker, a small loop of our own: the worker attempts the task, an
evaluator (the same local model, with structured output) checks the answer
against your success criteria, and if unmet, feedback goes back to the
worker to try again, up to 3 attempts.

The full implementation lives in `sidekick.py` and `sidekick_tools.py`,
next to this notebook — have a read of both before continuing.

## 0. Setup

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")
if IN_COLAB:
    print("Note: this lab launches local subprocess MCP servers (npx) and a Gradio app —")
    print("it works best on your local PC. On Colab, Parts 1-2 (below) still run fine.")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q langchain langgraph langchain-huggingface transformers torch accelerate duckduckgo-search wikipedia gradio langchain-mcp-adapters
else:
    %pip install -q langchain langgraph langchain-ollama ollama duckduckgo-search wikipedia gradio langchain-mcp-adapters


## One adjustment for Windows

When a stdio MCP server is launched from inside Jupyter on Windows, the
kernel's stderr has no real file descriptor and the launch fails. This cell
redirects the servers' error log so everything works. On Mac/Linux it does
nothing.


In [ ]:
import sys
if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions
    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, nothing to do here")


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from sidekick_tools import search, send_push_notification, wikipedia_lookup
from sidekick import Sidekick

if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=300)
    model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
else:
    from langchain_ollama import ChatOllama
    model = ChatOllama(model="llama3.2:3b", temperature=0.3)

print("model ready")


## Step 1: the simplest Sidekick, with no evaluator

At its plainest, the Sidekick is a `create_agent` with a few tools and
memory, plus a tiny function that runs one turn. No evaluator here — the
worker simply does its best. For many tasks, this simple version is all you
need.


In [ ]:
simple_worker = create_agent(
    model=model,
    tools=[search, send_push_notification, wikipedia_lookup],
    system_prompt="You are Sidekick, a helpful personal assistant. Use your tools to complete the task.",
    checkpointer=InMemorySaver(),
)

async def ask(worker, message):
    config = {"configurable": {"thread_id": "simple-sidekick"}}
    result = await worker.ainvoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].content

reply = await ask(simple_worker, "Search for who won the Nobel Prize in Physics in 2023 and send a notification with a short summary.")
print(reply)


## Step 2: human-in-the-loop with middleware

`HumanInTheLoopMiddleware` pauses the run and returns an interrupt describing
the pending action. We can approve, edit, reject, or respond, then resume.


In [ ]:
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Meeting booked with {person} on {day}."

approval_agent = create_agent(
    model=model,
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "approval-demo"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])


In [ ]:
resumed = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(resumed["messages"][-1].content)


### Choosing how to respond

Each decision can be **approve**, **edit** (run with changed arguments), or
**reject** (skip the tool, hand feedback back to the model):

```python
await approval_agent.ainvoke(
    Command(resume={"decisions": [{"type": "reject", "message": "No meetings on Fridays"}]}), config
)
```


In [ ]:
config = {"configurable": {"thread_id": "rejection-demo"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)
interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])


In [ ]:
resumed = await approval_agent.ainvoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Rejected - no meetings on Fridays"}]}), config)
print(resumed["messages"][-1].content)


## Step 3: the full Sidekick

The complete version lives in `sidekick.py` / `sidekick_tools.py`. The worker
has the full local toolkit: a headed browser and a sandboxed filesystem
through free MCP subprocess sessions, plus search, Wikipedia, and
notifications. The middleware stack from above is all in place, and the
evaluator loop checks each answer against your success criteria.

This is the whole stack in one object: the worker is a Layer 3
`create_agent`, its tools are Layer 1 `@tool` functions plus MCP servers, its
memory is a Layer 2 checkpointer, and the loop around it is code you wrote
yourself — and every part of it is free to run.


In [ ]:
sidekick = Sidekick(model)
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")


In [ ]:
history = await sidekick.run_turn(
    message="Go to Hacker News at news.ycombinator.com and tell me the title of the current top story.",
    success_criteria="The reply names a specific story currently on the Hacker News front page.",
    history=[],
)
for entry in history:
    print(f"[{entry['role']}] {entry['content'][:200]}\n")


## A real errand

A task worthy of the week: find the best flight. This needs a plan, a
stretch of real browsing, a file, and a notification, and it ends with the
Sidekick asking your permission.


In [ ]:
flight_task = """Find me the best round-trip flight from New York to London, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
itineraries with two or more stops. Write your recommendation with the top three options to flights.md,
then send me a notification with the price of your top pick."""

flight_criteria = "flights.md is written with three specific options including airline, times and price, plus a clear recommendation, and a notification was logged with the recommended price."

history = await sidekick.run_turn(flight_task, flight_criteria, history)
print(history[-1]["content"])


In [ ]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")


In [ ]:
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")


## The Gradio app

`app.py` wraps all of this in a chat interface: a request box, a success
criteria box, the Sidekick's plan updating live, and an Approve button. Run
it from a terminal with `python app.py`, or launch it right here.


In [ ]:
from app import ui, LAUNCH_STYLE
ui.launch(**LAUNCH_STYLE)


## Congratulations

You've built the Sidekick and travelled the whole local-model stack: the
building blocks of Layer 1, LangGraph orchestration, `create_agent` of Layer
3, the Deep Agents harness, and a real project mixing them all — with
middleware for planning, guardrails and human approval — running entirely on
free, open-weight models.

## Exercise
Make the Sidekick truly yours: set it a task you actually need done this
week, with a clear success criterion. For a bigger challenge, gate the
filesystem write tools behind the approval middleware in `sidekick.py`, and
add a tool of your own. Then compare: swap `llama3.2:3b` for `qwen2.5:7b` (if
your hardware allows) and see how much more reliably it completes multi-step
errands — a concrete lesson in the local-model capability ceiling.
